# FedAvg with Dirichlet Only (No Rotations)

This notebook implements FedAvg on CIFAR-10 with **ONLY label heterogeneity**:
- **NO feature heterogeneity**: All clients use same transformations (no rotations)
- **Label heterogeneity**: Dirichlet distribution with tunable alpha parameter

**Purpose:**
Baseline comparison to isolate the impact of label heterogeneity on FedAvg performance.

**Alpha parameter:**
- `alpha = 0.1` → Highly non-IID (extreme label imbalance per client)
- `alpha = 0.5` → Moderate non-IID
- `alpha = 1.0` → Balanced non-IID
- `alpha = 10.0` → Nearly IID

In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    !pip install -q torch torchvision scikit-learn matplotlib seaborn scipy
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

## Import Libraries

In [ ]:
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, f1_score
from sklearn.preprocessing import label_binarize
import copy
import random
import time
from collections import defaultdict
import itertools

sys.path.append('..')
from training.utils import get_model, set_seed

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load Configuration

In [ ]:
# Load configuration from JSON
import os
if os.path.exists('config.json'):
    config_path = 'config.json'
elif os.path.exists('experiments/config.json'):
    config_path = 'experiments/config.json'
else:
    raise FileNotFoundError("config.json not found")

with open(config_path, 'r') as f:
    CONFIG = json.load(f)

# Add Dirichlet alpha parameter
CONFIG['dirichlet_alpha'] = 0.5  # ← TUNE THIS: 0.1 (high non-IID) to 10.0 (low non-IID)

# Set random seeds
SEED = CONFIG['seed']
set_seed(SEED)
CONFIG['seed'] = SEED

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Load and Prepare Data

**No rotations applied** - only standard preprocessing and normalization.

In [ ]:
# Standard transform (no rotation)
standard_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10
print("Loading CIFAR-10 dataset...")
train_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
test_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=None)

print(f"Train dataset size: {len(train_dataset_raw)}")
print(f"Test dataset size: {len(test_dataset_raw)}")
print(f"✓ No rotation transformations applied - all clients use same preprocessing")

## Distribute Data with Dirichlet Only

**Only label heterogeneity** - Dirichlet distribution creates non-IID label distributions across clients.

In [ ]:
# Split ratios for train/val/test
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

print(f"Data split ratios: Train={train_ratio}, Val={val_ratio}, Test={test_ratio}")

# Organize data by class
num_classes = 10
indices_by_class = [[] for _ in range(num_classes)]

for idx, (_, label) in enumerate(train_dataset_raw):
    indices_by_class[label].append(idx)

print(f"\nTotal samples per class:")
for class_id in range(num_classes):
    print(f"  Class {class_id}: {len(indices_by_class[class_id])} samples")

# STEP 1: Split each class into train/val/test FIRST
train_indices_by_class = [[] for _ in range(num_classes)]
val_indices_by_class = [[] for _ in range(num_classes)]
test_indices_by_class = [[] for _ in range(num_classes)]

for class_id in range(num_classes):
    class_indices = np.array(indices_by_class[class_id])
    np.random.shuffle(class_indices)
    
    n = len(class_indices)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train_indices_by_class[class_id] = class_indices[:train_end].tolist()
    val_indices_by_class[class_id] = class_indices[train_end:val_end].tolist()
    test_indices_by_class[class_id] = class_indices[val_end:].tolist()

print(f"\nSplit samples per class:")
for class_id in range(num_classes):
    print(f"  Class {class_id}: Train={len(train_indices_by_class[class_id])}, Val={len(val_indices_by_class[class_id])}, Test={len(test_indices_by_class[class_id])}")

# STEP 2: Apply Dirichlet distribution to train and val ONLY (test is IID)
print(f"\nApplying Dirichlet distribution (alpha={CONFIG['dirichlet_alpha']}) to train and val...")
print(f"Test set will be IID (uniform distribution)")

def distribute_with_dirichlet(indices_by_class, num_clients, alpha):
    """Distribute data to clients using Dirichlet distribution."""
    client_indices = [[] for _ in range(num_clients)]
    
    for class_id in range(len(indices_by_class)):
        class_indices = np.array(indices_by_class[class_id])
        np.random.shuffle(class_indices)
        
        # Sample proportions from Dirichlet distribution
        proportions = np.random.dirichlet(alpha=[alpha] * num_clients)
        proportions = (np.cumsum(proportions) * len(class_indices)).astype(int)[:-1]
        
        # Split indices according to proportions
        splits = np.split(class_indices, proportions)
        
        for client_idx, split in enumerate(splits):
            client_indices[client_idx].extend(split.tolist())
    
    # Shuffle each client's indices
    for client_idx in range(num_clients):
        random.shuffle(client_indices[client_idx])
    
    return client_indices

def distribute_iid(indices_by_class, num_clients):
    """Distribute data uniformly (IID) across clients."""
    # Flatten all indices
    all_indices = []
    for class_indices in indices_by_class:
        all_indices.extend(class_indices)
    
    # Shuffle
    random.shuffle(all_indices)
    
    # Split uniformly
    client_indices = [[] for _ in range(num_clients)]
    samples_per_client = len(all_indices) // num_clients
    
    for client_idx in range(num_clients):
        start = client_idx * samples_per_client
        end = start + samples_per_client if client_idx < num_clients - 1 else len(all_indices)
        client_indices[client_idx] = all_indices[start:end]
    
    return client_indices

# Apply Dirichlet to train and val, IID to test
client_indices_train = distribute_with_dirichlet(train_indices_by_class, CONFIG['num_clients'], CONFIG['dirichlet_alpha'])
    client_indices_val = distribute_iid(val_indices_by_class, CONFIG['num_clients'])
client_indices_test = distribute_iid(test_indices_by_class, CONFIG['num_clients'])

# Create datasets for each client (no rotation transformation)
class StandardCIFAR10Dataset(Dataset):
    """CIFAR-10 dataset with standard preprocessing only."""
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        image = self.transform(image)
        return image, label

# Create standard dataset
standard_dataset = StandardCIFAR10Dataset(train_dataset_raw, standard_transform)

train_subsets = []
val_subsets = []
test_subsets = []
client_label_distributions = []

print(f"\nCreating datasets with standard preprocessing (no rotations)...")

for client_idx in range(CONFIG['num_clients']):
    # Create subsets
    train_subset = Subset(standard_dataset, client_indices_train[client_idx])
    val_subset = Subset(standard_dataset, client_indices_val[client_idx])
    test_subset = Subset(standard_dataset, client_indices_test[client_idx])
    
    train_subsets.append(train_subset)
    val_subsets.append(val_subset)
    test_subsets.append(test_subset)
    
    # Track label distribution for this client (using train data)
    labels = [train_dataset_raw[idx][1] for idx in client_indices_train[client_idx]]
    label_dist = np.bincount(labels, minlength=num_classes)
    client_label_distributions.append(label_dist)

print(f"\nCreated {len(train_subsets)} client train datasets")
print(f"Created {len(val_subsets)} client validation datasets")
print(f"Created {len(test_subsets)} client test datasets")
print(f"\nAverage samples per client:")
print(f"  Train: {np.mean([len(s) for s in train_subsets]):.1f}")
print(f"  Validation: {np.mean([len(s) for s in val_subsets]):.1f}")
print(f"  Test: {np.mean([len(s) for s in test_subsets]):.1f}")
print(f"\nTotal samples:")
print(f"  Train: {sum([len(s) for s in train_subsets])}")
print(f"  Validation: {sum([len(s) for s in val_subsets])}")
print(f"  Test: {sum([len(s) for s in test_subsets])}")
print(f"  Total: {sum([len(s) for s in train_subsets]) + sum([len(s) for s in val_subsets]) + sum([len(s) for s in test_subsets])}")
    print(f"\n✓ Train: Dirichlet (non-IID, alpha={CONFIG['dirichlet_alpha']})")
    print(f"✓ Val & Test: IID (uniform distribution)")


## Visualize Label Distribution

In [ ]:
# Visualize label distribution heterogeneity
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Samples per client
ax = axes[0, 0]
client_sizes = [len(subset) for subset in train_subsets]
ax.bar(range(CONFIG['num_clients']), client_sizes, alpha=0.7)
ax.axhline(y=np.mean(client_sizes), color='red', linestyle='--', label=f'Mean: {np.mean(client_sizes):.0f}')
ax.set_xlabel('Client ID')
ax.set_ylabel('Number of Samples')
ax.set_title(f'Data Distribution Across Clients\n(Std: {np.std(client_sizes):.1f})')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Label distribution heatmap
ax = axes[0, 1]
label_dist_matrix = np.array(client_label_distributions).T
im = ax.imshow(label_dist_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xlabel('Client ID')
ax.set_ylabel('Class Label')
ax.set_title(f'Label Distribution per Client\n(Dirichlet α={CONFIG["dirichlet_alpha"]})')
plt.colorbar(im, ax=ax, label='Sample Count')

# Plot 3: Classes per client histogram
ax = axes[1, 0]
classes_per_client = [np.sum(dist > 0) for dist in client_label_distributions]
ax.hist(classes_per_client, bins=range(1, 12), alpha=0.7, edgecolor='black')
ax.set_xlabel('Number of Classes Present')
ax.set_ylabel('Number of Clients')
ax.set_title(f'Classes per Client Distribution\n(Mean: {np.mean(classes_per_client):.2f})')
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Label entropy per client
ax = axes[1, 1]
entropies = []
for dist in client_label_distributions:
    probs = dist / dist.sum()
    probs = probs[probs > 0]
    entropy = -np.sum(probs * np.log2(probs))
    entropies.append(entropy)

ax.hist(entropies, bins=20, alpha=0.7, edgecolor='black')
ax.axhline(x=np.log2(num_classes), color='red', linestyle='--', label=f'Max (uniform): {np.log2(num_classes):.2f}')
ax.set_xlabel('Entropy (bits)')
ax.set_ylabel('Number of Clients')
ax.set_title(f'Label Distribution Entropy\n(Mean: {np.mean(entropies):.2f}, Lower = more imbalanced)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('fedavg_dirichlet_only_data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nLabel distribution statistics:")
print(f"  Mean entropy: {np.mean(entropies):.3f} bits")
print(f"  Max possible entropy: {np.log2(num_classes):.3f} bits (uniform)")
print(f"  Entropy std: {np.std(entropies):.3f}")

## Create Validation and Test Datasets

In [ ]:
# Create validation dataset
print("Creating validation dataset...")
val_dataset = torch.utils.data.ConcatDataset(val_subsets)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Validation dataset size: {len(val_dataset)}")

# Create test dataset
print("Creating test dataset...")
test_dataset = torch.utils.data.ConcatDataset(test_subsets)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Test dataset size: {len(test_dataset)}")
print(f"\n✓ All splits (train/val/test) have SAME label distributions")
print(f"⚠️  Test set will ONLY be evaluated at the end (proper ML practice)")

## Initialize Global Model

In [ ]:
# Initialize global model
global_model = get_model(
    model_name=CONFIG['model_name'],
    num_classes=10,
    pretrained=CONFIG['pretrained']
).to(device)

criterion = nn.CrossEntropyLoss()

print(f"Global model initialized: {CONFIG['model_name']}")

## Define FedAvg Functions

In [ ]:
def train_local_model(model, train_loader, epochs, lr):
    """Train local model for specified epochs."""
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    for epoch in range(epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    return model.state_dict()

def aggregate_models(client_weights, client_sizes):
    """Aggregate client models using weighted averaging."""
    total_size = sum(client_sizes)
    avg_weights = copy.deepcopy(client_weights[0])
    
    for key in avg_weights.keys():
        avg_weights[key] = torch.zeros_like(avg_weights[key], dtype=torch.float32)
        for i in range(len(client_weights)):
            weight = client_sizes[i] / total_size
            avg_weights[key] += client_weights[i][key] * weight
    
    return avg_weights

def evaluate_model(model, data_loader):
    """Evaluate model on given dataset."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_loss = total_loss / total
    accuracy = correct / total
    
    return avg_loss, accuracy

print("FedAvg functions defined")

## Run FedAvg Training

In [ ]:
# FedAvg configuration
local_epochs = CONFIG['local_epochs']
client_fraction = CONFIG['client_fraction']
num_selected = max(int(client_fraction * CONFIG['num_clients']), 1)

# Use fedavg_rounds if specified
if 'fedavg_rounds' in CONFIG:
    num_rounds = CONFIG['fedavg_rounds']
    print(f"Using configured FedAvg rounds: {num_rounds}")
else:
    # Calculate rounds to match ensemble budget
    ensemble_warmup = CONFIG['warmup_epochs'] * CONFIG['num_clients']
    ensemble_hierarchical = CONFIG.get('training_rounds', 30) * CONFIG['num_clients'] * CONFIG['local_epochs']
    ensemble_total = ensemble_warmup + ensemble_hierarchical
    
    num_rounds = int(ensemble_total / (num_selected * local_epochs))
    print(f"Calculated FedAvg rounds to match ensemble budget: {num_rounds}")

# Verify training budget
fedavg_total = num_rounds * num_selected * local_epochs
ensemble_warmup_budget = CONFIG['warmup_epochs'] * CONFIG['num_clients']
ensemble_hierarchical_budget = CONFIG['training_rounds'] * CONFIG['num_clients'] * CONFIG['local_epochs']
ensemble_total_budget = ensemble_warmup_budget + ensemble_hierarchical_budget

print(f"\nFedAvg Training Budget:")
print(f"  FedAvg: {num_rounds} rounds × {num_selected} clients × {local_epochs} epochs = {fedavg_total} client-epochs")
print(f"\nEnsemble Training Budget (for comparison):")
print(f"  Warmup: {CONFIG['warmup_epochs']} epochs × {CONFIG['num_clients']} clients = {ensemble_warmup_budget} client-epochs")
print(f"  Hierarchical: {CONFIG['training_rounds']} rounds × {CONFIG['num_clients']} clients × {CONFIG['local_epochs']} epochs = {ensemble_hierarchical_budget} client-epochs")
print(f"  Total: {ensemble_total_budget} client-epochs")

# Storage for results
val_losses = []
val_accs = []
round_times = []

print(f"\n{'='*70}")
print(f"FEDAVG TRAINING (Dirichlet α={CONFIG['dirichlet_alpha']}, NO ROTATIONS)")
print(f"{'='*70}")
print(f"Number of rounds: {num_rounds}")
print(f"Local epochs: {local_epochs}")
print(f"Clients per round: {num_selected}/{CONFIG['num_clients']} ({client_fraction*100:.0f}%)")
print(f"{'='*70}\n")

total_start_time = time.time()

for round_num in range(1, num_rounds + 1):
    round_start = time.time()
    
    # Sample clients for this round
    selected_clients = random.sample(range(CONFIG['num_clients']), num_selected)
    
    # Store client updates
    client_weights = []
    client_sizes = []
    
    # Get global weights
    global_weights = global_model.state_dict()
    
    # Train selected clients
    for client_idx in selected_clients:
        # Create local model
        local_model = get_model(
            model_name=CONFIG['model_name'],
            num_classes=10,
            pretrained=False
        ).to(device)
        local_model.load_state_dict(global_weights)
        
        # Create data loader
        train_loader = DataLoader(
            train_subsets[client_idx],
            batch_size=CONFIG['batch_size'],
            shuffle=True
        )
        
        # Train locally
        updated_weights = train_local_model(local_model, train_loader, local_epochs, CONFIG['lr'])
        
        # Store update
        client_weights.append(updated_weights)
        client_sizes.append(len(train_subsets[client_idx]))
    
    # Aggregate updates
    aggregated_weights = aggregate_models(client_weights, client_sizes)
    global_model.load_state_dict(aggregated_weights)
    
    # Evaluate on VALIDATION set
    val_loss, val_acc = evaluate_model(global_model, val_loader)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    round_time = time.time() - round_start
    round_times.append(round_time)
    
    # Print progress
    if round_num % 5 == 0 or round_num == 1:
        print(f"Round {round_num}/{num_rounds} - "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, "
              f"Time: {round_time:.2f}s")

total_training_time = time.time() - total_start_time

print(f"\n{'='*70}")
print(f"FedAvg Training Complete!")
print(f"{'='*70}")
print(f"Total training time: {total_training_time:.2f}s ({total_training_time/60:.2f} min)")
print(f"Average time per round: {np.mean(round_times):.2f}s")
print(f"Final validation accuracy: {val_accs[-1]:.4f}")
print(f"Best validation accuracy: {max(val_accs):.4f} (round {np.argmax(val_accs)+1})")

## Final Test Set Evaluation

In [ ]:
# Final evaluation on TEST set (only once!)
print(f"{'='*70}")
print("FINAL TEST SET EVALUATION")
print(f"{'='*70}\n")

final_test_loss, final_test_acc = evaluate_model(global_model, test_loader)

print(f"Final Test Loss: {final_test_loss:.4f}")
print(f"Final Test Accuracy: {final_test_acc:.4f}")
print(f"\nComparison:")
print(f"  Best Validation Accuracy: {max(val_accs):.4f} (round {np.argmax(val_accs)+1})")
print(f"  Final Test Accuracy: {final_test_acc:.4f}")
print(f"  Difference (Val - Test): {max(val_accs) - final_test_acc:+.4f}")
print(f"{'='*70}\n")

In [ ]:
# Collect predictions and labels on test set
print("Collecting predictions on test set...")
global_model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = global_model(inputs)
        
        # Get probabilities for AUC
        probs = torch.nn.functional.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"\n{'='*70}")
print("DETAILED TEST SET METRICS")
print(f"{'='*70}\n")

# 1. Classification Report
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# 2. Overall Metrics
macro_f1 = f1_score(all_labels, all_preds, average='macro')
micro_f1 = f1_score(all_labels, all_preds, average='micro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"\nF1-Score Summary:")
print(f"  Macro F1:    {macro_f1:.4f}")
print(f"  Micro F1:    {micro_f1:.4f}")
print(f"  Weighted F1: {weighted_f1:.4f}")

# 3. ROC-AUC Score
try:
    labels_binarized = label_binarize(all_labels, classes=range(10))
    macro_auc = roc_auc_score(labels_binarized, all_probs, average='macro')
    weighted_auc = roc_auc_score(labels_binarized, all_probs, average='weighted')
    per_class_auc = roc_auc_score(labels_binarized, all_probs, average=None)
    
    print(f"\nROC-AUC Summary:")
    print(f"  Macro AUC:    {macro_auc:.4f}")
    print(f"  Weighted AUC: {weighted_auc:.4f}")
    print(f"\nPer-class AUC:")
    for i, name in enumerate(class_names):
        print(f"  {name:12s}: {per_class_auc[i]:.4f}")
except Exception as e:
    print(f"\nNote: Could not compute AUC scores: {e}")

print(f"\n{'='*70}\n")

## Confusion Matrix Visualization

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Counts
ax = axes[0]
im1 = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im1, ax=ax)
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       xlabel='Predicted Label',
       ylabel='True Label',
       title=f'Confusion Matrix - Counts\nFedAvg Dirichlet Only (α={CONFIG["dirichlet_alpha"]})')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm[i, j], 'd'),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=9)

# Plot 2: Percentages
ax = axes[1]
im2 = ax.imshow(cm_percent, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im2, ax=ax, format='%.1f%%')
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       xlabel='Predicted Label',
       ylabel='True Label',
       title=f'Confusion Matrix - Percentages\nFedAvg Dirichlet Only (α={CONFIG["dirichlet_alpha"]})')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

thresh = cm_percent.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm_percent[i, j], '.1f'),
            ha="center", va="center",
            color="white" if cm_percent[i, j] > thresh else "black",
            fontsize=9)

plt.tight_layout()
plt.savefig(f'fedavg_dirichlet_only_alpha{CONFIG["dirichlet_alpha"]}_confusion_matrix.png', 
            dpi=300, bbox_inches='tight')
plt.show()

print(f"Confusion matrix saved")

# Per-class accuracy
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
print(f"\nPer-class Accuracy:")
for i, name in enumerate(class_names):
    print(f"  {name:12s}: {per_class_accuracy[i]:.4f} ({per_class_accuracy[i]*100:.2f}%)")

## Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Validation loss
ax = axes[0]
rounds_range = range(1, num_rounds + 1)
ax.plot(rounds_range, val_losses, 'o-', linewidth=2, markersize=4)
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Validation Loss', fontsize=12)
ax.set_title(f'FedAvg Validation Loss (α={CONFIG["dirichlet_alpha"]}, No Rotations)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 2: Validation accuracy
ax = axes[1]
ax.plot(rounds_range, val_accs, 's-', linewidth=2, markersize=4, color='green', label='Validation')
ax.axhline(y=max(val_accs), color='red', linestyle='--', alpha=0.5, 
           label=f'Best Val: {max(val_accs):.4f}')
ax.axhline(y=final_test_acc, color='blue', linestyle='--', alpha=0.5,
           label=f'Final Test: {final_test_acc:.4f}')
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title(f'FedAvg Validation Accuracy (α={CONFIG["dirichlet_alpha"]}, No Rotations)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(f'fedavg_dirichlet_only_alpha{CONFIG["dirichlet_alpha"]}_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Training curves saved")

## Save Results

In [ ]:
# Save results
results = {
    'method': 'fedavg_dirichlet_only',
    'config': CONFIG,
    'heterogeneity': 'label_only',  # NEW: Mark heterogeneity type
    'num_rounds': num_rounds,
    'local_epochs': local_epochs,
    'client_fraction': client_fraction,
    'dirichlet_alpha': CONFIG['dirichlet_alpha'],
    'times': {
        'training_time': total_training_time,
        'avg_round_time': float(np.mean(round_times))
    },
    'performance': {
        'best_val_acc': float(max(val_accs)),
        'best_val_round': int(np.argmax(val_accs) + 1),
        'final_val_acc': float(val_accs[-1]),
        'final_test_acc': float(final_test_acc),
        'final_test_loss': float(final_test_loss),
        'val_test_gap': float(max(val_accs) - final_test_acc)
    },
    'val_losses': [float(x) for x in val_losses],
    'val_accs': [float(x) for x in val_accs],
    'test_metrics': {
        'macro_f1': float(macro_f1),
        'micro_f1': float(micro_f1),
        'weighted_f1': float(weighted_f1),
        'macro_auc': float(macro_auc) if 'macro_auc' in locals() else None,
        'weighted_auc': float(weighted_auc) if 'weighted_auc' in locals() else None,
        'per_class_auc': [float(x) for x in per_class_auc] if 'per_class_auc' in locals() else None,
        'per_class_accuracy': [float(x) for x in per_class_accuracy],
        'confusion_matrix': cm.tolist()
    },
    'data_stats': {
        'mean_train_samples_per_client': float(np.mean([len(s) for s in train_subsets])),
        'std_train_samples_per_client': float(np.std([len(s) for s in train_subsets])),
        'mean_val_samples_per_client': float(np.mean([len(s) for s in val_subsets])),
        'total_train_samples': sum([len(s) for s in train_subsets]),
        'total_val_samples': sum([len(s) for s in val_subsets]),
        'mean_label_entropy': float(np.mean(entropies)),
        'std_label_entropy': float(np.std(entropies))
    }
}

filename = f'fedavg_dirichlet_only_alpha{CONFIG["dirichlet_alpha"]}_results.json'
with open(filename, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to '{filename}'")

# Save model checkpoint
checkpoint_name = f'fedavg_dirichlet_only_alpha{CONFIG["dirichlet_alpha"]}_checkpoint.pth'
torch.save({
    'round': num_rounds,
    'model_state_dict': global_model.state_dict(),
    'best_val_acc': max(val_accs),
    'final_test_acc': final_test_acc,
    'config': CONFIG
}, checkpoint_name)

print(f"Model checkpoint saved to '{checkpoint_name}'")

## Summary

**FedAvg with Label Heterogeneity Only:**

**Key Characteristics:**
- **NO rotation-based feature heterogeneity**: All clients use same preprocessing
- **ONLY label heterogeneity**: Dirichlet distribution creates non-IID label distributions
- **Global model**: Single model trained via federated averaging

**Expected Behavior:**
- Lower alpha → More conflicting gradients → Lower accuracy
- Higher alpha → More balanced → Higher accuracy
- Should show sensitivity to alpha (unlike pooled ensemble)

**Comparison Goals:**
- vs **FedAvg (Rotation + Dirichlet)**: Impact of dual vs single heterogeneity
- vs **Ensemble (Dirichlet Only)**: Flat vs hierarchical architecture for label heterogeneity
- Across alpha values: Measure of FedAvg's robustness to label imbalance

**Research Value:**
Isolates label heterogeneity impact on federated learning without confounding feature heterogeneity.